# 13 — RQ2 final build (cross-community prevalence)

Loads the scored RQ2 corpus (unseen manosphere-adjacent creators + three comparison communities), 
applies the nested 1B rule, and estimates construct prevalence per community with a hierarchical 
bootstrap (channels within community, videos within channel) and channel-level permutation tests. 
Cells 6–8 render the reporting figures from the saved results.

**Prerequisite:** the scored corpus `rq2_FINAL_primary_12mo_SCORED.csv` must exist in `outputs/rq2/` 
(produced by applying the frozen RQ1 models from notebook 10 to the RQ2 comments). Results are 
written under `outputs/rq2/rq2_prevalence/`.


In [ ]:
# 0. IMPORTS, CONFIG, CONSTANTS
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = PROJECT_ROOT / "outputs" / "rq2"

# Scored RQ2 corpus: unseen manosphere-adjacent creators + comparison communities
SCORED = RESULTS / "rq2_FINAL_primary_12mo_SCORED.csv"
OUT    = RESULTS / "rq2_prevalence"
OUT.mkdir(parents=True, exist_ok=True)

COMMUNITIES = [
    "manosphere_adjacent",
    "male_oriented_wellbeing",
    "mainstream_self_improvement",
    "unrelated_control",
]
COMPARATORS = COMMUNITIES[1:]

B_BOOT = 10_000
B_PERM = 100_000
SEED = 42

In [ ]:
# 1. LOAD + APPLY THE 1B NESTING
def load_and_constrain(scored_path):
    df = pd.read_csv(
        scored_path, low_memory=False,
        dtype={"comment_id": "string", "channel_name": "string", "video_id": "string"},
    )
    needed = ["community_group", "channel_name", "video_id", "pred_1A", "pred_1B", "pred_2"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{Path(scored_path).name} missing columns: {missing}")

    for c in ["pred_1A", "pred_1B", "pred_2"]:
        v = pd.to_numeric(df[c], errors="raise")
        assert v.isin([0, 1]).all(), f"{c} is not binary"
        df[c] = v.astype(int)

    assert set(df["community_group"]) == set(COMMUNITIES)

    # 1B is nested in 1A: a comment is 1B only where 1A is also positive
    df["pred_1B"] = ((df["pred_1A"] == 1) & (df["pred_1B"] == 1)).astype(int)
    assert (df["pred_1B"] <= df["pred_1A"]).all()
    return df

In [ ]:
# 2. COMMUNITY PREVALENCE
def community_prevalence(df):
    cols = {
        "prevalence_1A": "pred_1A",
        "prevalence_1B": "pred_1B",
        "prevalence_2": "pred_2",
    }
    video = df.groupby(
        ["community_group", "channel_name", "video_id"], as_index=False
    ).agg(**{k: (v, "mean") for k, v in cols.items()})

    channel = video.groupby(
        ["community_group", "channel_name"], as_index=False
    ).agg(**{k: (k, "mean") for k in cols})

    community = channel.groupby("community_group")[list(cols)].mean()

    for frame in (video, channel):
        assert (frame["prevalence_1B"] <= frame["prevalence_1A"] + 1e-12).all()
    return video, channel, community

In [ ]:
# 3. HIERARCHICAL BOOTSTRAP (resample channels within community, then videos within channel)
def bootstrap_cis(video, point, seed=SEED, B=B_BOOT):
    rng = np.random.default_rng(seed)
    constructs = ["1A", "1B", "2"]
    boot = {c: {com: np.empty(B) for com in COMMUNITIES} for c in constructs}

    for b in range(B):
        for community in COMMUNITIES:
            cv = video[video["community_group"] == community]
            channels = cv["channel_name"].unique()
            sampled_channels = rng.choice(channels, size=len(channels), replace=True)
            for con in constructs:
                col = f"prevalence_{con}"
                ch_means = []
                for ch in sampled_channels:
                    vals = cv.loc[cv["channel_name"] == ch, col].to_numpy()
                    samp = rng.choice(vals, size=len(vals), replace=True)
                    ch_means.append(samp.mean())
                boot[con][community][b] = np.mean(ch_means)

    community_rows, contrast_rows = [], []
    for con in constructs:
        for community in COMMUNITIES:
            vals = boot[con][community]
            community_rows.append({
                "construct": con, "community": community,
                "prevalence_percent": point.loc[community, f"prevalence_{con}"] * 100,
                "ci_low_percent": np.percentile(vals, 2.5) * 100,
                "ci_high_percent": np.percentile(vals, 97.5) * 100,
            })
        mano_pt = point.loc["manosphere_adjacent", f"prevalence_{con}"]
        for comp in COMPARATORS:
            diff = boot[con]["manosphere_adjacent"] - boot[con][comp]
            contrast_rows.append({
                "construct": con,
                "comparison": f"manosphere_adjacent - {comp}",
                "difference_pp": (mano_pt - point.loc[comp, f"prevalence_{con}"]) * 100,
                "ci_low_pp": np.percentile(diff, 2.5) * 100,
                "ci_high_pp": np.percentile(diff, 97.5) * 100,
            })
    return pd.DataFrame(community_rows), pd.DataFrame(contrast_rows)

In [ ]:
 # 4. CHANNEL-LEVEL PERMUTATION TESTS
def omnibus_stat(values, labels):
    overall = np.mean(values); stat = 0.0
    for g in np.unique(labels):
        vals = values[labels == g]
        stat += len(vals) * (np.mean(vals) - overall) ** 2
    return stat

def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float); m = len(pvals)
    order = np.argsort(pvals); adjusted = np.empty(m); running_max = 0.0
    for rank, idx in enumerate(order):
        running_max = max(running_max, (m - rank) * pvals[idx])
        adjusted[idx] = min(running_max, 1.0)
    return adjusted

def permutation_tests(channel, seed=SEED, B=B_PERM):
    assert channel["channel_name"].nunique() == 16
    assert (channel.groupby("community_group")["channel_name"].nunique() == 4).all()
    cc = {"1A": "prevalence_1A", "1B": "prevalence_1B", "2": "prevalence_2"}
    rng = np.random.default_rng(seed)
    labels0 = channel["community_group"].to_numpy()

    omni = []
    for con, col in cc.items():
        values = channel[col].to_numpy()
        observed = omnibus_stat(values, labels0)
        perm = np.empty(B)
        for b in range(B):
            perm[b] = omnibus_stat(values, rng.permutation(labels0))
        omni.append({"construct": con, "observed_statistic": observed,
                     "permutation_p": (1 + np.sum(perm >= observed)) / (B + 1)})

    pw = []
    for con, col in cc.items():
        mano = channel.loc[channel["community_group"] == "manosphere_adjacent", col].to_numpy()
        for comp in COMPARATORS:
            other = channel.loc[channel["community_group"] == comp, col].to_numpy()
            combined = np.concatenate([mano, other]); obs = np.mean(mano) - np.mean(other)
            diffs = []
            for idx in itertools.combinations(range(len(combined)), len(mano)):
                mask = np.zeros(len(combined), dtype=bool); mask[list(idx)] = True
                diffs.append(combined[mask].mean() - combined[~mask].mean())
            diffs = np.array(diffs)
            pw.append({"construct": con, "comparison": f"manosphere_adjacent - {comp}",
                       "difference_pp": obs * 100,
                       "exact_p": float(np.mean(np.abs(diffs) >= abs(obs)))})
    pw = pd.DataFrame(pw); pw["holm_p"] = np.nan
    for con in cc:
        m = pw["construct"] == con
        pw.loc[m, "holm_p"] = holm_adjust(pw.loc[m, "exact_p"].to_numpy())
    return pd.DataFrame(omni), pw

In [ ]:
# 5. RUN RQ2 END-TO-END
def run_rq2(scored_path=SCORED, out_dir=OUT):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    print("=" * 78); print("RQ2 PREVALENCE"); print("=" * 78)

    df = load_and_constrain(scored_path)
    video, channel, community = community_prevalence(df)

    community_ci, contrasts = bootstrap_cis(video, community)
    print("\nCOMMUNITY PREVALENCE WITH 95% BOOTSTRAP CIs:")
    print(community_ci.round(2).to_string(index=False))
    print("\nMANOSPHERE CONTRASTS (pp, 95% CI):")
    print(contrasts.round(2).to_string(index=False))

    omni, pw = permutation_tests(channel)
    print("\nOMNIBUS PERMUTATION TESTS:"); print(omni.round(5).to_string(index=False))
    print("\nPAIRWISE (exact) + HOLM:"); print(pw.round(5).to_string(index=False))

    df.to_csv(out_dir / "rq2_scored.csv", index=False, encoding="utf-8-sig")
    channel.to_csv(out_dir / "rq2_channel_results.csv", index=False)
    community.to_csv(out_dir / "rq2_community_results.csv")
    community_ci.to_csv(out_dir / "rq2_bootstrap_community_CI.csv", index=False)
    contrasts.to_csv(out_dir / "rq2_bootstrap_contrasts.csv", index=False)
    omni.to_csv(out_dir / "rq2_permutation_omnibus.csv", index=False)
    pw.to_csv(out_dir / "rq2_permutation_pairwise.csv", index=False)
    print(f"\nSaved to: {out_dir}\n")
    return community_ci, contrasts, omni, pw

community_ci, contrasts, omni, pw = run_rq2()

In [ ]:
# ============================================================
# 6. REPORT FIGURE — COMMUNITY PREVALENCE AND 95% CIs
# Uses the saved results, including constrained 1B.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. LOAD SAVED RESULTS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq2" / "rq2_prevalence"

INPUT_PATH = OUTPUT_DIR / "rq2_bootstrap_community_CI.csv"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Saved results not found:\n{INPUT_PATH}\n"
        "Run the RQ2 analysis cell first."
    )

results = pd.read_csv(
    INPUT_PATH,
    dtype={"construct": "string", "community": "string"}
)

community_order = [
    "manosphere_adjacent",
    "male_oriented_wellbeing",
    "mainstream_self_improvement",
    "unrelated_control",
]

community_labels = [
    "Manosphere-adjacent",
    "Male-oriented wellbeing",
    "Mainstream self-improvement",
    "Unrelated control",
]

panels = [
    ("1A", "1A · Grievance"),
    ("1B", "1B · Weaponised victimhood"),
    ("2", "2 · Conspiratorial framing"),
]

numeric_columns = [
    "prevalence_percent",
    "ci_low_percent",
    "ci_high_percent",
]

required = {"construct", "community", *numeric_columns}
missing = required - set(results.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

if results.duplicated(["construct", "community"]).any():
    raise ValueError("Duplicate construct–community results found.")

expected_pairs = {
    (construct, community)
    for construct, _ in panels
    for community in community_order
}

observed_pairs = set(
    results[["construct", "community"]]
    .itertuples(index=False, name=None)
)

if observed_pairs != expected_pairs:
    raise ValueError("Expected three constructs across four communities.")

for column in numeric_columns:
    results[column] = pd.to_numeric(results[column], errors="raise")

if not np.isfinite(results[numeric_columns].to_numpy()).all():
    raise ValueError("Missing or non-finite estimates or intervals.")

if not results[numeric_columns].apply(
    lambda column: column.between(0, 100)
).all().all():
    raise ValueError("Expected percentage values between 0 and 100.")

if (results["ci_low_percent"] > results["ci_high_percent"]).any():
    raise ValueError("A confidence interval has reversed endpoints.")

# ------------------------------------------------------------
# 2. CREATE THREE PANELS WITH A SHARED PERCENTAGE SCALE
# ------------------------------------------------------------

navy = "#213E64"
blue = "#6685A5"

# Automatically include the highest estimate or interval endpoint.
largest_value = results[numeric_columns].to_numpy().max()
axis_max = max(10, np.ceil(largest_value / 10) * 10)

with plt.rc_context({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
    "pdf.fonttype": 42,
}):
    fig, axes = plt.subplots(
        1, 3,
        figsize=(12, 4.4),
        sharex=True,
        sharey=True
    )

    for ax, (construct, title) in zip(axes, panels):
        subset = (
            results.loc[results["construct"] == construct]
            .set_index("community")
            .loc[community_order]
        )

        for row_number, (_, row) in enumerate(subset.iterrows()):
            estimate = row["prevalence_percent"]
            lower = row["ci_low_percent"]
            upper = row["ci_high_percent"]

            colour = navy if row_number == 0 else blue

            # Draw interval and endpoint caps.
            ax.hlines(
                row_number, lower, upper,
                color=colour, linewidth=1.8
            )
            ax.vlines(
                [lower, upper],
                row_number - 0.075,
                row_number + 0.075,
                color=colour, linewidth=1.5
            )

            # Draw point estimate.
            ax.plot(
                estimate, row_number,
                marker="o", markersize=6.5,
                color=colour
            )

            # Exact percentage above each point.
            ax.annotate(
                f"{estimate:.2f}%",
                xy=(estimate, row_number),
                xytext=(0, 12),
                textcoords="offset points",
                ha="center",
                fontsize=9,
                color=navy
            )

        ax.set_title(title, fontsize=10, pad=20)
        ax.set_xlim(-2, axis_max + 2)
        ax.set_xticks(np.arange(0, axis_max + 1, 10))
        ax.set_ylim(3.5, -0.65)

        ax.grid(axis="x", color="#E7EBEF", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.tick_params(axis="y", length=0)

        for spine in ["top", "right", "left"]:
            ax.spines[spine].set_visible(False)

        ax.spines["bottom"].set_color("#B9C2CC")

    axes[0].set_yticks(range(4), community_labels)

    fig.supxlabel(
        "Model-detected prevalence (%)",
        fontsize=11,
        y=0.04
    )

    fig.subplots_adjust(
        left=0.245,
        right=0.985,
        top=0.80,
        bottom=0.19,
        wspace=0.20
    )

    # --------------------------------------------------------
    # 3. SAVE
    # --------------------------------------------------------

    png_path = OUTPUT_DIR / "rq2_community_prevalence.png"
    pdf_path = OUTPUT_DIR / "rq2_community_prevalence.pdf"

    fig.savefig(
        png_path,
        dpi=350,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
# ============================================================
# RQ2 — PORTRAIT PREVALENCE FIGURE
# Uses saved prevalence estimates and bootstrap intervals.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. LOAD SAVED RESULTS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq2" / "rq2_prevalence"

INPUT_PATH = OUTPUT_DIR / "rq2_bootstrap_community_CI.csv"

results = pd.read_csv(
    INPUT_PATH,
    dtype={"construct": str, "community": str}
)

required = [
    "construct", "community", "prevalence_percent",
    "ci_low_percent", "ci_high_percent"
]

missing = set(required) - set(results.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

communities = [
    "manosphere_adjacent",
    "male_oriented_wellbeing",
    "mainstream_self_improvement",
    "unrelated_control",
]

labels = [
    "Manosphere-adjacent",
    "Male-oriented wellbeing",
    "Mainstream self-improvement",
    "Unrelated control",
]

panels = [
    ("1A", "1A · Grievance/victimhood"),
    ("1B", "1B · Weaponised victimhood"),
    ("2", "2 · Conspiratorial framing"),
]

if results.duplicated(["construct", "community"]).any():
    raise ValueError("Duplicate construct–community results found.")

results = results.set_index(["construct", "community"])

expected = pd.MultiIndex.from_product(
    [[p[0] for p in panels], communities],
    names=["construct", "community"]
)

if not expected.isin(results.index).all():
    raise ValueError("Some expected construct–community results are missing.")

results = results.loc[expected].copy()

numeric_columns = [
    "prevalence_percent", "ci_low_percent", "ci_high_percent"
]
results[numeric_columns] = results[numeric_columns].apply(
    pd.to_numeric, errors="raise"
)

values = results[numeric_columns].to_numpy()
if not np.isfinite(values).all():
    raise ValueError("Missing or non-finite estimates found.")

if ((values < 0) | (values > 100)).any():
    raise ValueError("Expected percentages between 0 and 100.")

if (results["ci_low_percent"] > results["ci_high_percent"]).any():
    raise ValueError("A confidence interval has reversed bounds.")

# ------------------------------------------------------------
# 2. DRAW THREE VERTICALLY STACKED PANELS
# ------------------------------------------------------------

NAVY = "#213E64"
BLUE = "#6685A5"
GRID = "#E7EBEF"

# Common scale preserves comparisons across constructs.
upper = max(
    results["ci_high_percent"].max(),
    results["prevalence_percent"].max()
)
axis_max = max(10, int(np.ceil(upper / 10) * 10))

with plt.rc_context({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}):
    fig, axes = plt.subplots(
        3, 1,
        figsize=(7.2, 7.4),
        sharex=True
    )

    for ax, (construct, title) in zip(axes, panels):
        data = results.xs(
            construct, level="construct"
        ).loc[communities]

        for y, community in enumerate(communities):
            row = data.loc[community]
            estimate = row["prevalence_percent"]
            low = row["ci_low_percent"]
            high = row["ci_high_percent"]

            colour = NAVY if y == 0 else BLUE

            # Confidence interval and end caps
            ax.hlines(y, low, high, color=colour, linewidth=1.8)
            ax.vlines(
                [low, high], y - 0.07, y + 0.07,
                color=colour, linewidth=1.4
            )

            # Point estimate
            ax.plot(
                estimate, y, "o",
                color=colour, markersize=6, zorder=3
            )

            # Percentage above each point
            ax.annotate(
                f"{estimate:.2f}%",
                xy=(estimate, y),
                xytext=(0, 8),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9,
                color=colour
            )

        ax.set_title(title, loc="left", pad=12)
        ax.set_yticks(range(4))
        ax.set_yticklabels(labels, fontsize=9.5)
        ax.set_ylim(3.45, -0.65)

        ax.set_xlim(-1.5, axis_max + 1.5)
        ax.set_xticks(np.arange(0, axis_max + 1, 10))
        ax.tick_params(axis="x", labelbottom=True, labelsize=9)
        ax.tick_params(axis="y", length=0, pad=8)

        ax.set_axisbelow(True)
        ax.grid(axis="x", color=GRID, linewidth=0.8)

        for spine in ["top", "right", "left"]:
            ax.spines[spine].set_visible(False)

        ax.spines["bottom"].set_color("#B9C3CF")

    axes[-1].set_xlabel(
        "Model-detected prevalence (%)",
        fontsize=10,
        labelpad=10
    )

    fig.subplots_adjust(
        left=0.35,
        right=0.97,
        top=0.95,
        bottom=0.09,
        hspace=0.52
    )

    # --------------------------------------------------------
    # 3. SAVE
    # --------------------------------------------------------

    for extension in ["png", "pdf"]:
        output_path = (
            OUTPUT_DIR /
            f"rq2_community_prevalence_portrait.{extension}"
        )
        fig.savefig(
            output_path,
            dpi=350,
            bbox_inches="tight",
            facecolor="white"
        )
        print("Saved:", output_path)

    plt.show()

In [ ]:
# ============================================================
# RQ2 — COMMUNITY PREVALENCE FIGURE
# Three stacked panels using saved estimates and intervals.
# No analyses are rerun.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator

# ------------------------------------------------------------
# 1. LOAD SAVED RESULTS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq2" / "rq2_prevalence"

INPUT_PATH = OUTPUT_DIR / "rq2_bootstrap_community_CI.csv"

results = pd.read_csv(
    INPUT_PATH,
    dtype={"construct": str, "community": str}
)

required = [
    "construct",
    "community",
    "prevalence_percent",
    "ci_low_percent",
    "ci_high_percent",
]

missing = set(required) - set(results.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

communities = [
    "manosphere_adjacent",
    "male_oriented_wellbeing",
    "mainstream_self_improvement",
    "unrelated_control",
]

labels = [
    "Manosphere-adjacent",
    "Male-oriented wellbeing",
    "Mainstream self-improvement",
    "Unrelated control",
]

# Construct, heading, axis maximum, tick spacing
panels = [
    ("1A", "Grievance/victimhood", 40, 10),
    ("1B", "Weaponised victimhood", 10, 2),
    ("2", "Conspiratorial framing", 10, 2),
]

if results.duplicated(["construct", "community"]).any():
    raise ValueError("Duplicate construct–community results found.")

results = results.set_index(["construct", "community"])

expected = pd.MultiIndex.from_product(
    [[panel[0] for panel in panels], communities],
    names=["construct", "community"]
)

if not expected.isin(results.index).all():
    raise ValueError("Expected construct–community results are missing.")

results = results.loc[expected].copy()

numeric_columns = [
    "prevalence_percent",
    "ci_low_percent",
    "ci_high_percent",
]

results[numeric_columns] = results[numeric_columns].apply(
    pd.to_numeric, errors="raise"
)

values = results[numeric_columns].to_numpy()

if not np.isfinite(values).all():
    raise ValueError("Missing or non-finite estimates found.")

if ((values < 0) | (values > 100)).any():
    raise ValueError("Expected percentages between 0 and 100.")

if (results["ci_low_percent"] > results["ci_high_percent"]).any():
    raise ValueError("A confidence interval has reversed bounds.")

# ------------------------------------------------------------
# 2. DRAW FIGURE
# ------------------------------------------------------------

NAVY = "#203B56"
BLUE = "#637F96"
TEXT = "#252525"
GRID = "#E6E9EC"

sns.set_theme(style="white", context="paper")

with plt.rc_context({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Arial", "Liberation Sans", "DejaVu Sans"
    ],
    "font.size": 10.5,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": "#555555",
    "ytick.color": TEXT,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}):
    fig, axes = plt.subplots(
        3, 1,
        figsize=(7.5, 6.2),
        sharex=False
    )

    for ax, (construct, title, axis_max, tick_step) in zip(
        axes, panels
    ):
        data = results.xs(
            construct, level="construct"
        ).loc[communities]

        # Subtle shading for the manosphere cohort.
        ax.axhspan(
            -0.40, 0.40,
            color="#F0F4F7",
            zorder=0
        )

        for y, community in enumerate(communities):
            row = data.loc[community]

            estimate = float(row["prevalence_percent"])
            low = float(row["ci_low_percent"])
            high = float(row["ci_high_percent"])

            # Prevent silently clipping any confidence interval.
            if not (
                0 <= low <= high <= axis_max
                and 0 <= estimate <= axis_max
            ):
                raise ValueError(
                    f"{construct}, {community}: estimate or "
                    f"interval falls outside 0–{axis_max}%."
                )

            colour = NAVY if y == 0 else BLUE

            # Confidence interval
            ax.plot(
                [low, high],
                [y, y],
                color=colour,
                linewidth=2,
                solid_capstyle="round",
                zorder=2
            )

            # Point estimate
            ax.scatter(
                estimate, y,
                s=38,
                color=colour,
                edgecolor="white",
                linewidth=0.6,
                zorder=3,
                clip_on=False
            )

            # Percentages aligned in a separate column.
            ax.text(
                1.055, y,
                f"{estimate:.2f}%",
                transform=ax.get_yaxis_transform(),
                ha="left",
                va="center",
                fontsize=11,
                fontweight="bold" if y == 0 else "normal",
                color=colour,
                clip_on=False
            )

        ax.text(
            -0.64, 1.15,
            f"{construct}   {title}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=12,
            fontweight="bold"
        )

        ax.set_yticks(range(4))
        ax.set_yticklabels(labels, fontsize=10.5)
        ax.set_ylim(3.50, -0.55)

        # Different scales, clearly labelled in each panel.
        ax.set_xlim(0, axis_max)
        ax.xaxis.set_major_locator(MultipleLocator(tick_step))

        ax.tick_params(
            axis="x",
            labelsize=9.5,
            length=0,
            pad=5
        )
        ax.tick_params(axis="y", length=0, pad=12)

        ax.set_axisbelow(True)
        ax.grid(axis="x", color=GRID, linewidth=0.7)
        ax.grid(axis="y", visible=False)

        for spine in ax.spines.values():
            spine.set_visible(False)

    axes[-1].set_xlabel(
        "Model-detected prevalence (%)",
        fontsize=11,
        labelpad=10
    )

    fig.subplots_adjust(
        left=0.39,
        right=0.88,
        top=0.92,
        bottom=0.10,
        hspace=0.68
    )

    # --------------------------------------------------------
    # 3. SAVE PNG AND PDF
    # --------------------------------------------------------

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for extension in ["png", "pdf"]:
        path = OUTPUT_DIR / (
            f"rq2_prevalence_publication.{extension}"
        )
        fig.savefig(
            path,
            dpi=400,
            facecolor="white"
        )
        print("Saved:", path)

    plt.show()
